In [8]:
# --- Section 1: Setup & Data Loading ---
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample

# Load datasets based on your uploaded files
# Note: Ensure the CSV files are in the same directory
df_train = pd.read_csv('UNSW_NB15_training-set.csv')
df_test = pd.read_csv('UNSW_NB15_testing-set.csv')

# --- Section 2: Feature Engineering (Interrelation Logic) ---
# Train directly on attack_cat so the app can show real threat families
drop_cols = ['id', 'proto', 'service', 'state', 'attack_cat']
X = df_train.drop(drop_cols + ['label'], axis=1)
y = df_train['attack_cat']

# Stratify so the rare classes like Worms remain represented in validation
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# --- Section 3: Targeted Rare-Class Rebalancing ---
# Oversample the weakest families so the classifier sees more Analysis, Backdoor, Worms, and Shellcode examples
train_df = X_train.copy()
train_df['attack_cat'] = y_train.values
resample_targets = {
    'Analysis': 4000,
    'Backdoor': 4000,
    'Worms': 2000,
    'Shellcode': 2500,
}
augmented_parts = [train_df]
for attack_name, target_size in resample_targets.items():
    subset = train_df[train_df['attack_cat'] == attack_name]
    if len(subset) < target_size:
        augmented_parts.append(
            resample(
                subset,
                replace=True,
                n_samples=target_size - len(subset),
                random_state=32,
            )
        )
augmented_train_df = pd.concat(augmented_parts, ignore_index=True)
X_train_balanced = augmented_train_df.drop(columns=['attack_cat'])
y_train_balanced = augmented_train_df['attack_cat']

# --- Section 4: Robust Pipeline Construction ---
# Balanced random forest plus targeted oversampling gives better recall on the rarest families
ids_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=160,
        max_depth=40,
        class_weight='balanced_subsample',
        n_jobs=1,
        random_state=32,
    ))
])

# Training on multiclass data after targeted rare-class rebalancing
ids_pipeline.fit(X_train_balanced, y_train_balanced)
val_pred = ids_pipeline.predict(X_val)
print('Training Complete.')
print(f"Validation accuracy: {accuracy_score(y_val, val_pred):.4f}")
print(classification_report(y_val, val_pred, zero_division=0))

# --- Section 5: Model Export ---
model_filename = 'military_ids_model.pkl'
joblib.dump(ids_pipeline, model_filename)
# Export feature names so the UI knows which inputs to ask for
joblib.dump(X_train_balanced.columns.tolist(), 'feature_names.pkl')
print(f"Model and features successfully exported to {model_filename}")
print(f"Exported classes: {list(ids_pipeline.classes_)}")